In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, recall_score, balanced_accuracy_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.feature_selection import mutual_info_classif
from skrebate import ReliefF
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
import warnings
from collections import defaultdict, Counter
warnings.filterwarnings('ignore')

K_FEATURES = 2

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='random_forest', k=10, feature_names=None):
        self.method = method
        self.k = k
        self.feature_names = feature_names
        self.selected_indices_ = None
        self._selector = None

    def fit(self, X, y=None):
        n_features = X.shape[1]
        self.k = min(self.k, n_features)

        if self.method == 'rfe':
            estimator = SVC(kernel='linear', random_state=42)
            self._selector = RFE(estimator=estimator, n_features_to_select=self.k, step=1)
            self._selector.fit(X, y)
            self.selected_indices_ = np.where(self._selector.support_)[0]

        elif self.method == 'mrmr':
            std = np.std(X, axis=0)
            valid_mask = std > 1e-10
            if not valid_mask.all():
                X_filtered = X[:, valid_mask]
                original_indices = np.where(valid_mask)[0]
            else:
                X_filtered = X
                original_indices = np.arange(n_features)

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_filtered)

            try:
                rel = mutual_info_classif(X_scaled, y, random_state=42)
            except:
                rel, _ = f_classif(X_scaled, y)

            corr = np.abs(np.corrcoef(X_scaled.T))
            np.fill_diagonal(corr, 0)

            rel_norm = rel / (rel.max() if rel.max() > 0 else 1)
            corr_norm = corr / (corr.max() if corr.max() > 0 else 1)

            selected = []
            remaining = list(range(X_filtered.shape[1]))
            for _ in range(self.k):
                best_score = -np.inf
                best_idx = None
                for i in remaining:
                    red_mean = corr_norm[i, selected].mean() if selected else 0
                    score = rel_norm[i] - red_mean
                    if score > best_score:
                        best_score = score
                        best_idx = i
                if best_idx is not None:
                    selected.append(best_idx)
                    remaining.remove(best_idx)

            self.selected_indices_ = original_indices[np.array(selected)]

        elif self.method == 'relieff':
            n_neighbors = min(10, max(1, len(y) // 3))
            self._selector = ReliefF(n_neighbors=n_neighbors, n_features_to_select=self.k)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.top_features_[:self.k]

        elif self.method == 'lasso':
            estimator = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000)
            self._selector = SelectFromModel(estimator=estimator, max_features=self.k, threshold=-np.inf)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.get_support(indices=True)

        elif self.method == 'random_forest':
            estimator = RandomForestClassifier(n_estimators=100, random_state=42)
            self._selector = SelectFromModel(estimator=estimator, max_features=self.k, threshold=-np.inf)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.get_support(indices=True)

        else:
            raise ValueError("Unknown method")
        return self

    def transform(self, X):
        return X[:, self.selected_indices_]

def load_and_merge_data():
    # User: modify these paths to your actual data file locations
    path_coupling = "path"
    path_traditional = "path"
    
    raw_id_col = 'Dataset' 
    
    try:
        data1 = pd.read_excel(path_coupling)
        data2 = pd.read_excel(path_traditional)
        
        print(f"Cross-frequency data loaded: {data1.shape}")
        print(f"Traditional parameters data loaded: {data2.shape}")
        
        def clean_subject_id(raw_id):
            s = str(raw_id)
            s = s.replace('.set', '')
            core_id = s.split('_')[0]
            return core_id

        data1['Cleaned_ID'] = data1[raw_id_col].apply(clean_subject_id)
        data2['Cleaned_ID'] = data2[raw_id_col].apply(clean_subject_id)
        
        data_merged = pd.merge(
            data1, 
            data2, 
            on='Cleaned_ID', 
            how='inner', 
            suffixes=('', '_trad')
        )
        
        print(f"Merge completed: {data_merged.shape}")
        return data_merged
        
    except Exception as e:
        print(f"Error loading files: {e}")
        import traceback
        traceback.print_exc()
        print("\nGenerating dummy data for demonstration...")
        from sklearn.datasets import make_classification
        X_dummy, y_dummy = make_classification(n_samples=100, n_features=30, n_informative=5, random_state=42)
        data = pd.DataFrame(X_dummy, columns=[f'Feature_{i:02d}' for i in range(30)])
        data['Group'] = ['MDD' if i == 1 else 'HC' for i in y_dummy]
        data['Condition'] = 'EC'
        data['Band'] = 'Alpha' 
        data['Cleaned_ID'] = range(100)
        return data

def create_feature_set(data, condition, feature_columns):
    subset = data[data['Condition'] == condition].copy()
    
    if 'Band' in subset.columns and 'Alpha' in subset['Band'].values:
        subset = subset[subset['Band'] == 'Alpha'].copy()

    if subset.empty:
        return None, None, None
    
    available_cols = [c for c in feature_columns if c in subset.columns]
    
    missing_cols = set(feature_columns) - set(available_cols)
    if missing_cols:
        print(f"Warning: missing columns: {missing_cols}")

    X = subset[available_cols].copy()
    y = (subset['Group'] == 'MDD').astype(int).values
    
    return X.values, y, available_cols

def calculate_specificity(y_true, y_pred):
    from sklearn.metrics import confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

scoring = {
    'balanced_acc': 'balanced_accuracy',
    'accuracy': 'accuracy',
    'sensitivity': make_scorer(recall_score),
    'specificity': make_scorer(calculate_specificity),
    'roc_auc': 'roc_auc'
}

features_coupling = [
    'Broadband-Delta deltaP_row_CA', 'Broadband-Delta deltaP_col_AC'
]

features_traditional = [
    'IndExpVar_A', 'MeanDuration_B', 'Coverage_A',
    'DeltaTM_D->A'
]

all_feature_columns = features_coupling + features_traditional

data = load_and_merge_data()

X, y, current_feature_names = create_feature_set(data, 'EC', all_feature_columns)
if X is None:
    raise ValueError("No data available")

algorithms_to_test = ['rfe', 'mrmr', 'relieff', 'lasso', 'random_forest']

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
k_features_range = [K_FEATURES]

all_results = []
feature_selection_records = defaultdict(list)
best_params_records = defaultdict(list)
best_estimators_records = defaultdict(list)

print(f"Fixed number of features: {K_FEATURES}")

for algo_name in algorithms_to_test:
    print(f"Processing: {algo_name.upper()}")
    
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
        ('selector', FeatureSelector(method=algo_name, k=K_FEATURES, feature_names=current_feature_names)),
        ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    
    param_grid = {
        'selector__k': k_features_range,
        'classifier__n_estimators': [100, 150, 200],
        'classifier__max_depth': [2, 5, 10, None],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [2, 5, 10],
        'classifier__max_features': ['sqrt', 'log2'],
        'classifier__class_weight': ['balanced']
    }
    
    grid = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='balanced_accuracy', n_jobs=-1)
    nested_scores = cross_validate(grid, X, y, cv=outer_cv, scoring=scoring, n_jobs=-1, return_estimator=True)
    
    result_row = {
        'Algorithm': algo_name,
        'Balanced_Acc': f"{np.mean(nested_scores['test_balanced_acc']):.3f} ± {np.std(nested_scores['test_balanced_acc']):.3f}",
        'Accuracy': f"{np.mean(nested_scores['test_accuracy']):.3f} ± {np.std(nested_scores['test_accuracy']):.3f}",
        'Sensitivity': f"{np.mean(nested_scores['test_sensitivity']):.3f} ± {np.std(nested_scores['test_sensitivity']):.3f}",
        'Specificity': f"{np.mean(nested_scores['test_specificity']):.3f} ± {np.std(nested_scores['test_specificity']):.3f}",
        'AUC': f"{np.mean(nested_scores['test_roc_auc']):.3f} ± {np.std(nested_scores['test_roc_auc']):.3f}",
    }
    all_results.append(result_row)
    
    for fold_idx, estimator in enumerate(nested_scores['estimator']):
        best_params_records[algo_name].append(estimator.best_params_)
        best_estimators_records[algo_name].append(estimator.best_estimator_)
        selected_idx = estimator.best_estimator_.named_steps['selector'].selected_indices_
        selected_names = [current_feature_names[i] for i in selected_idx]
        feature_selection_records[algo_name].extend(selected_names)

results_df = pd.DataFrame(all_results)
column_order = ['Algorithm', 'Balanced_Acc', 'Accuracy', 'Sensitivity', 'Specificity', 'AUC']
results_df = results_df[column_order]

def extract_mean(val_str):
    try:
        return float(val_str.split('±')[0])
    except:
        return 0

results_df['Sort_Key'] = results_df['Balanced_Acc'].apply(extract_mean)
results_df = results_df.sort_values('Sort_Key', ascending=False).drop('Sort_Key', axis=1)

best_algo = results_df.iloc[0]['Algorithm']
best_result = results_df.iloc[0:1]

print(best_result.to_string(index=False))

print(f"Best algorithm: {best_algo.upper()}")
param_list = [str(p) for p in best_params_records[best_algo]]
most_common_param = Counter(param_list).most_common(1)[0][0]
print(f"Best params: {most_common_param}")

cnt = Counter(feature_selection_records[best_algo])
most_common_features = cnt.most_common(10)
print("Feature selection frequency (Top 10):")
for feat_name, count in most_common_features:
    print(f"  {count:2d} | {feat_name}")

feature_importance_dict = defaultdict(float)
best_estimators = best_estimators_records[best_algo]

for pipe in best_estimators:
    selector = pipe.named_steps['selector']
    selected_idx = selector.selected_indices_
    selected_feat_names = [current_feature_names[i] for i in selected_idx]
    rf = pipe.named_steps['classifier']
    importances = rf.feature_importances_
    for name, imp in zip(selected_feat_names, importances):
        feature_importance_dict[name] += imp

n_folds = len(best_estimators)
for key in feature_importance_dict:
    feature_importance_dict[key] /= n_folds

sorted_importances = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

print("Mean feature importance:")
for rank, (feat_name, imp_value) in enumerate(sorted_importances, 1):
    print(f"  Rank {rank:2d} | {imp_value:.4f} | {feat_name}")